In [ ]:
# -*- coding: utf-8 -*-
"""
Example 4: 混合车型 + 模拟退火求解
"""

from collections import Counter
from math import exp
from pathlib import Path
import random
import time

from py3dbp import Packer, Bin, Item

TRUCK_TYPES = {
    "type1": {
        "label": "车型1",
        "WHD": (420, 210, 217),
        "max_weight": 6000,
        "cost": 450,
        "corner": 0,
        "put_type": 0,
    },
    "type2": {
        "label": "车型2",
        "WHD": (680, 245, 247),
        "max_weight": 10000,
        "cost": 600,
        "corner": 0,
        "put_type": 0,
    },
}
TRUCK_TYPE_KEYS = tuple(TRUCK_TYPES.keys())

FIX_POINT = True
CHECK_STABLE = True
SUPPORT_SURFACE_RATIO = 0.75
BIGGER_FIRST = True
NUMBER_OF_DECIMALS = 0

RUN_SORT_SWITCH_VERIFICATION = False
RUN_ALL_BASELINES = False

RANDOM_SEED = 20260418
SA_RUNS = 3
SA_ITERATIONS = 45
INITIAL_TEMPERATURE = 1200.0
COOLING_RATE = 0.2
INITIAL_PERTURB_STEPS = 4

UNFITTED_COUNT_PENALTY = 20000.0
UNFITTED_VOLUME_PENALTY = 0.01
UNFITTED_WEIGHT_PENALTY = 5.0
ASSIGNMENT_PENALTY = 50000.0
TRUCK_LIMIT_PENALTY = 100000.0
MAX_TRUCK_COUNT = 28
OBJECTIVE_MODE = "min_cost"
OBJECTIVE_MODE_LABELS = {
    "min_trucks": "车辆最少",
    "min_cost": "成本最低",
}
MAX_TOTAL_COST = MAX_TRUCK_COUNT * max(truck["cost"] for truck in TRUCK_TYPES.values())
TRUCK_COUNT_PRIORITY = MAX_TOTAL_COST + 1
COST_PRIORITY = MAX_TRUCK_COUNT + 1
CONSTRAINT_PRIORITY = 100

SHOW_PLOTS = False
SAVE_PLOTS = False
PLOT_OUTPUT_DIR = Path.cwd() / "example"

EVALUATION_CACHE = {}
PACK_TRUCK_CACHE = {}


def add_items(item_specs, prefix, name, whd, weight, count, loadbear, updown, color, level=1, typeof="cube"):
    for index in range(count):
        item_id = len(item_specs)
        item_specs.append({
            "id": item_id,
            "partno": f"{prefix}{index + 1}",
            "name": name,
            "typeof": typeof,
            "WHD": whd,
            "weight": weight,
            "level": level,
            "loadbear": loadbear,
            "updown": updown,
            "color": color,
        })


def build_item_specs():
    item_specs = []
    add_items(item_specs, "3.68L*4", "3.68L*4", (31, 31, 35), 4, 100, 0, True, "#F31212")
    add_items(item_specs, "1.8L*6", "1.8L*6", (35, 35, 31), 2, 100, 0.05, True, "#F312C2")
    add_items(item_specs, "4L*4", "4L*4", (32, 27, 37), 4, 100, 0.05, True, "#4312F3")
    add_items(item_specs, "5L*4常规品", "5L*4常规品", (33, 33, 39), 5, 100, 0.05, False, "#12F325")
    add_items(item_specs, "20L软", "20L软", (31, 18, 50), 18, 100, 0, False, "#12F3A8") 
    return item_specs


ITEM_SPECS = build_item_specs()
ITEM_BY_ID = {item["id"]: item for item in ITEM_SPECS}
PARTNO_TO_ID = {item["partno"]: item["id"] for item in ITEM_SPECS}
ALL_ITEM_IDS = [item["id"] for item in ITEM_SPECS]


def item_volume(item_id):
    width, height, depth = ITEM_BY_ID[item_id]["WHD"]
    return width * height * depth


def item_weight(item_id):
    return ITEM_BY_ID[item_id]["weight"]


def build_item(item_id):
    spec = ITEM_BY_ID[item_id]
    return Item(
        partno=spec["partno"],
        name=spec["name"],
        typeof=spec["typeof"],
        WHD=spec["WHD"],
        weight=spec["weight"],
        level=spec["level"],
        loadbear=spec["loadbear"],
        updown=spec["updown"],
        color=spec["color"],
    )


def build_box(truck_type, truck_index):
    truck = TRUCK_TYPES[truck_type]
    return Bin(
        partno=f"{truck_type}-{truck_index}",
        WHD=truck["WHD"],
        max_weight=truck["max_weight"],
        corner=truck["corner"],
        put_type=truck["put_type"],
    )


def default_item_order(item_ids=None):
    if item_ids is None:
        item_ids = ALL_ITEM_IDS
    return sorted(
        item_ids,
        key=lambda item_id: (
            ITEM_BY_ID[item_id]["level"],
            -ITEM_BY_ID[item_id]["loadbear"],
            -item_volume(item_id),
            -item_weight(item_id),
            item_id,
        ),
    )


def normalize_fleet(fleet):
    normalized = []
    for truck in fleet:
        item_ids = list(truck["item_ids"])
        if item_ids:
            normalized.append({
                "truck_type": truck["truck_type"],
                "item_ids": item_ids,
            })
    return normalized


def copy_fleet(fleet):
    return normalize_fleet(fleet)


def fleet_signature(fleet):
    normalized = normalize_fleet(fleet)
    return tuple(sorted((truck["truck_type"], tuple(truck["item_ids"])) for truck in normalized))


def assignment_penalty(fleet):
    assigned_ids = []
    for truck in fleet:
        assigned_ids.extend(truck["item_ids"])
    seen_ids = set(assigned_ids)
    missing_count = len(set(ALL_ITEM_IDS) - seen_ids)
    duplicate_count = len(assigned_ids) - len(seen_ids)
    penalty = (missing_count + duplicate_count) * ASSIGNMENT_PENALTY
    return penalty, missing_count, duplicate_count


def pack_truck(item_ids, truck_type, truck_index=1, sort_items=False, include_box=False):
    cache_key = (
        truck_type,
        tuple(item_ids),
        sort_items,
        FIX_POINT,
        CHECK_STABLE,
        SUPPORT_SURFACE_RATIO,
        NUMBER_OF_DECIMALS,
    )
    if not include_box and cache_key in PACK_TRUCK_CACHE:
        return PACK_TRUCK_CACHE[cache_key]

    truck = TRUCK_TYPES[truck_type]
    packer = Packer()
    box = build_box(truck_type, truck_index)
    packer.addBin(box)

    for item_id in item_ids:
        packer.addItem(build_item(item_id))

    packer.pack(
        bigger_first=BIGGER_FIRST,
        distribute_items=False,
        fix_point=FIX_POINT,
        check_stable=CHECK_STABLE,
        support_surface_ratio=SUPPORT_SURFACE_RATIO,
        number_of_decimals=NUMBER_OF_DECIMALS,
        sort_items=sort_items,
    )

    fitted_ids = [PARTNO_TO_ID[item.partno] for item in box.items]
    unfitted_ids = [PARTNO_TO_ID[item.partno] for item in box.unfitted_items]
    fitted_volume = sum(item_volume(item_id) for item_id in fitted_ids)
    fitted_weight = sum(item_weight(item_id) for item_id in fitted_ids)
    assigned_volume = sum(item_volume(item_id) for item_id in item_ids)
    assigned_weight = sum(item_weight(item_id) for item_id in item_ids)
    box_volume = truck["WHD"][0] * truck["WHD"][1] * truck["WHD"][2]
    placement_signature = tuple(sorted(
        (
            PARTNO_TO_ID[item.partno],
            tuple(int(value) for value in item.position),
            item.rotation_type,
        )
        for item in box.items
    ))

    result = {
        "truck_type": truck_type,
        "truck_label": truck["label"],
        "assigned_ids": list(item_ids),
        "fitted_ids": fitted_ids,
        "unfitted_ids": unfitted_ids,
        "assigned_count": len(item_ids),
        "fitted_count": len(fitted_ids),
        "unfitted_count": len(unfitted_ids),
        "assigned_volume": assigned_volume,
        "fitted_volume": fitted_volume,
        "unfitted_volume": assigned_volume - fitted_volume,
        "assigned_weight": assigned_weight,
        "fitted_weight": fitted_weight,
        "unfitted_weight": assigned_weight - fitted_weight,
        "space_utilization": round(fitted_volume / box_volume * 100, 2) if box_volume else 0,
        "load_utilization": round(fitted_weight / truck["max_weight"] * 100, 2) if truck["max_weight"] else 0,
        "cost": truck["cost"],
        "placement_signature": placement_signature,
        "item_mix": Counter(ITEM_BY_ID[item_id]["name"] for item_id in fitted_ids),
    }
    if include_box:
        result["box"] = box
        return result

    PACK_TRUCK_CACHE[cache_key] = result
    return result


def pack_fingerprint(result):
    return (
        tuple(sorted(result["fitted_ids"])),
        tuple(sorted(result["unfitted_ids"])),
        result["placement_signature"],
    )


def get_objective_label(objective_mode):
    return OBJECTIVE_MODE_LABELS.get(objective_mode, objective_mode)


def penalty_summary(total_unfitted_count, total_unfitted_volume, total_unfitted_weight, extra_penalty, truck_limit_exceeded):
    penalty = (
        total_unfitted_count * UNFITTED_COUNT_PENALTY
        + total_unfitted_volume * UNFITTED_VOLUME_PENALTY
        + total_unfitted_weight * UNFITTED_WEIGHT_PENALTY
        + extra_penalty
    )
    if truck_limit_exceeded:
        penalty += (MAX_TRUCK_COUNT + 1) * TRUCK_LIMIT_PENALTY
    return penalty


def compute_objective(total_cost, truck_count, penalty, objective_mode):
    if objective_mode == "min_cost":
        return penalty * COST_PRIORITY + total_cost
    if objective_mode == "min_trucks":
        return penalty * CONSTRAINT_PRIORITY * TRUCK_COUNT_PRIORITY + truck_count * TRUCK_COUNT_PRIORITY + total_cost
    raise ValueError(f"未知目标模式: {objective_mode}")


def evaluate_fleet(fleet, include_boxes=False, objective_mode=OBJECTIVE_MODE):
    fleet = normalize_fleet(fleet)
    signature = fleet_signature(fleet)
    cache_key = (objective_mode, signature)
    if not include_boxes and cache_key in EVALUATION_CACHE:
        return EVALUATION_CACHE[cache_key]

    total_cost = 0
    total_unfitted_count = 0
    total_unfitted_volume = 0
    total_unfitted_weight = 0
    truck_results = []
    truck_count = len(fleet)
    truck_limit_exceeded = truck_count > MAX_TRUCK_COUNT

    if not truck_limit_exceeded:
        for truck_index, truck in enumerate(fleet, start=1):
            truck_result = pack_truck(
                truck["item_ids"],
                truck["truck_type"],
                truck_index=truck_index,
                sort_items=False,
                include_box=include_boxes,
            )
            truck_results.append(truck_result)
            total_cost += truck_result["cost"]
            total_unfitted_count += truck_result["unfitted_count"]
            total_unfitted_volume += truck_result["unfitted_volume"]
            total_unfitted_weight += truck_result["unfitted_weight"]
    else:
        total_cost = sum(TRUCK_TYPES[truck["truck_type"]]["cost"] for truck in fleet)

    extra_penalty, missing_count, duplicate_count = assignment_penalty(fleet)
    penalty = penalty_summary(
        total_unfitted_count,
        total_unfitted_volume,
        total_unfitted_weight,
        extra_penalty,
        truck_limit_exceeded,
    )
    objective = compute_objective(total_cost, truck_count, penalty, objective_mode)

    result = {
        "fleet": copy_fleet(fleet),
        "objective": objective,
        "objective_mode": objective_mode,
        "objective_label": get_objective_label(objective_mode),
        "total_cost": total_cost,
        "truck_count": truck_count,
        "truck_type_counts": Counter(truck["truck_type"] for truck in fleet),
        "truck_results": truck_results,
        "total_unfitted_count": total_unfitted_count,
        "total_unfitted_volume": total_unfitted_volume,
        "total_unfitted_weight": total_unfitted_weight,
        "missing_count": missing_count,
        "duplicate_count": duplicate_count,
        "penalty": penalty,
        "truck_limit_exceeded": truck_limit_exceeded,
        "feasible": not truck_limit_exceeded and total_unfitted_count == 0 and missing_count == 0 and duplicate_count == 0,
    }

    if not include_boxes:
        EVALUATION_CACHE[cache_key] = result
    return result


def build_sequential_fleet(item_order, allowed_types):
    remaining_ids = list(item_order)
    fleet = []

    while remaining_ids:
        candidate_results = []
        for truck_type in allowed_types:
            result = pack_truck(remaining_ids, truck_type, truck_index=len(fleet) + 1, sort_items=False)
            if result["fitted_count"] > 0:
                candidate_results.append(result)

        if not candidate_results:
            item_id = remaining_ids[0]
            single_candidates = []
            for truck_type in allowed_types:
                result = pack_truck([item_id], truck_type, truck_index=len(fleet) + 1, sort_items=False)
                if result["fitted_count"] == 1:
                    single_candidates.append(result)
            if not single_candidates:
                raise ValueError(f"物品 {ITEM_BY_ID[item_id]['partno']} 无法装入任何车型")
            chosen = min(single_candidates, key=lambda result: result["cost"])
        else:
            chosen = max(
                candidate_results,
                key=lambda result: (
                    result["fitted_volume"] / result["cost"],
                    result["fitted_weight"] / result["cost"],
                    result["fitted_count"],
                    -result["cost"],
                ),
            )

        fitted_set = set(chosen["fitted_ids"])
        fleet.append({
            "truck_type": chosen["truck_type"],
            "item_ids": list(chosen["fitted_ids"]),
        })
        remaining_ids = [item_id for item_id in remaining_ids if item_id not in fitted_set]

    return normalize_fleet(fleet)


def build_baselines(objective_mode=OBJECTIVE_MODE):
    item_order = default_item_order()
    baseline_specs = [
        {"name": "混合贪心", "allowed_types": TRUCK_TYPE_KEYS},
    ]
    if RUN_ALL_BASELINES:
        baseline_specs = [
            {"name": "只用车型1", "allowed_types": ("type1",)},
            {"name": "只用车型2", "allowed_types": ("type2",)},
            {"name": "混合贪心", "allowed_types": TRUCK_TYPE_KEYS},
        ]

    baselines = []
    for spec in baseline_specs:
        fleet = build_sequential_fleet(item_order, spec["allowed_types"])
        baselines.append({
            "name": spec["name"],
            "fleet": fleet,
            "evaluation": evaluate_fleet(fleet, objective_mode=objective_mode),
        })

    return baselines


def move_item(fleet, rng):
    source_indexes = [index for index, truck in enumerate(fleet) if truck["item_ids"]]
    if not source_indexes:
        return None

    source_index = rng.choice(source_indexes)
    source_truck = fleet[source_index]
    item_index = rng.randrange(len(source_truck["item_ids"]))
    item_id = source_truck["item_ids"].pop(item_index)

    destination_indexes = [index for index in range(len(fleet)) if index != source_index]
    if destination_indexes and rng.random() < 0.75:
        destination_index = rng.choice(destination_indexes)
        destination_truck = fleet[destination_index]
        insert_index = rng.randrange(len(destination_truck["item_ids"]) + 1)
        destination_truck["item_ids"].insert(insert_index, item_id)
    else:
        fleet.append({"truck_type": rng.choice(TRUCK_TYPE_KEYS), "item_ids": [item_id]})

    return normalize_fleet(fleet)


def swap_items(fleet, rng):
    truck_indexes = [index for index, truck in enumerate(fleet) if truck["item_ids"]]
    if len(truck_indexes) < 2:
        return None

    first_index, second_index = rng.sample(truck_indexes, 2)
    first_truck = fleet[first_index]
    second_truck = fleet[second_index]
    first_item_index = rng.randrange(len(first_truck["item_ids"]))
    second_item_index = rng.randrange(len(second_truck["item_ids"]))
    first_truck["item_ids"][first_item_index], second_truck["item_ids"][second_item_index] = (
        second_truck["item_ids"][second_item_index],
        first_truck["item_ids"][first_item_index],
    )
    return normalize_fleet(fleet)


def reorder_truck(fleet, rng):
    eligible_indexes = [index for index, truck in enumerate(fleet) if len(truck["item_ids"]) >= 2]
    if not eligible_indexes:
        return None

    truck = fleet[rng.choice(eligible_indexes)]
    item_ids = truck["item_ids"]
    operation = rng.choice(("swap", "insert", "reverse"))

    if operation == "swap":
        first_index, second_index = sorted(rng.sample(range(len(item_ids)), 2))
        item_ids[first_index], item_ids[second_index] = item_ids[second_index], item_ids[first_index]
    elif operation == "insert":
        from_index = rng.randrange(len(item_ids))
        item_id = item_ids.pop(from_index)
        to_index = rng.randrange(len(item_ids) + 1)
        item_ids.insert(to_index, item_id)
    else:
        start_index, end_index = sorted(rng.sample(range(len(item_ids)), 2))
        item_ids[start_index:end_index + 1] = reversed(item_ids[start_index:end_index + 1])

    return normalize_fleet(fleet)


def change_truck_type(fleet, rng):
    if not fleet:
        return None

    truck = fleet[rng.randrange(len(fleet))]
    truck["truck_type"] = "type1" if truck["truck_type"] == "type2" else "type2"
    return normalize_fleet(fleet)


def split_truck(fleet, rng):
    eligible_indexes = [index for index, truck in enumerate(fleet) if len(truck["item_ids"]) >= 2]
    if not eligible_indexes:
        return None

    truck_index = rng.choice(eligible_indexes)
    truck = fleet[truck_index]
    item_ids = truck["item_ids"]
    start_index = rng.randrange(len(item_ids) - 1)
    end_index = rng.randrange(start_index + 1, len(item_ids) + 1)
    new_item_ids = item_ids[start_index:end_index]
    truck["item_ids"] = item_ids[:start_index] + item_ids[end_index:]
    fleet.append({"truck_type": rng.choice(TRUCK_TYPE_KEYS), "item_ids": new_item_ids})
    return normalize_fleet(fleet)


def merge_trucks(fleet, rng):
    if len(fleet) < 2:
        return None

    first_index, second_index = sorted(rng.sample(range(len(fleet)), 2))
    first_truck = fleet[first_index]
    second_truck = fleet[second_index]
    insert_index = rng.randrange(len(first_truck["item_ids"]) + 1)
    first_truck["item_ids"][insert_index:insert_index] = second_truck["item_ids"]
    del fleet[second_index]
    return normalize_fleet(fleet)


def generate_neighbor(fleet, rng):
    base_signature = fleet_signature(fleet)
    moves = [move_item, swap_items, reorder_truck, change_truck_type, split_truck, merge_trucks]

    for _ in range(20):
        candidate = copy_fleet(fleet)
        move = rng.choice(moves)
        candidate = move(candidate, rng)
        if candidate is None:
            continue
        if len(candidate) > MAX_TRUCK_COUNT:
            continue
        if fleet_signature(candidate) != base_signature:
            return candidate

    return copy_fleet(fleet)


def perturb_fleet(fleet, rng, steps):
    candidate = copy_fleet(fleet)
    for _ in range(steps):
        next_candidate = generate_neighbor(candidate, rng)
        if len(next_candidate) <= MAX_TRUCK_COUNT:
            candidate = next_candidate
    return candidate


def accept_candidate(current_objective, candidate_objective, temperature, rng):
    delta = candidate_objective - current_objective
    if delta <= 0:
        return True
    return rng.random() < exp(-delta / max(temperature, 1e-9))


def simulated_annealing(initial_fleet, seed, objective_mode=OBJECTIVE_MODE):
    rng = random.Random(seed)
    current_fleet = copy_fleet(initial_fleet)
    if len(current_fleet) > MAX_TRUCK_COUNT:
        raise ValueError(f"初始方案车数超过上限 {MAX_TRUCK_COUNT}")
    if INITIAL_PERTURB_STEPS > 0:
        current_fleet = perturb_fleet(current_fleet, rng, INITIAL_PERTURB_STEPS)

    current_evaluation = evaluate_fleet(current_fleet, objective_mode=objective_mode)
    best_fleet = copy_fleet(current_fleet)
    best_evaluation = current_evaluation
    temperature = INITIAL_TEMPERATURE

    for _ in range(SA_ITERATIONS):
        candidate_fleet = generate_neighbor(current_fleet, rng)
        if len(candidate_fleet) > MAX_TRUCK_COUNT:
            temperature *= COOLING_RATE
            continue

        candidate_evaluation = evaluate_fleet(candidate_fleet, objective_mode=objective_mode)

        if accept_candidate(current_evaluation["objective"], candidate_evaluation["objective"], temperature, rng):
            current_fleet = candidate_fleet
            current_evaluation = candidate_evaluation

        if current_evaluation["objective"] < best_evaluation["objective"]:
            best_fleet = copy_fleet(current_fleet)
            best_evaluation = current_evaluation

        temperature *= COOLING_RATE

    return {
        "seed": seed,
        "best_fleet": best_fleet,
        "best_evaluation": best_evaluation,
    }


def verify_sort_switch():
    rng = random.Random(RANDOM_SEED)
    candidate_sizes = (35, 50, 65)

    for truck_type in TRUCK_TYPE_KEYS:
        for sample_size in candidate_sizes:
            probe_ids = default_item_order()[:sample_size]
            reference_sorted = pack_truck(probe_ids, truck_type, sort_items=True)
            reversed_sorted = pack_truck(list(reversed(probe_ids)), truck_type, sort_items=True)
            reference_unsorted = pack_truck(probe_ids, truck_type, sort_items=False)
            reversed_unsorted = pack_truck(list(reversed(probe_ids)), truck_type, sort_items=False)

            if pack_fingerprint(reference_unsorted) != pack_fingerprint(reversed_unsorted):
                return {
                    "verified": True,
                    "truck_type": truck_type,
                    "sample_size": sample_size,
                    "changed_without_sort": True,
                    "method": "reversed",
                    "sorted_fitted_reference": reference_sorted["fitted_count"],
                    "sorted_fitted_reordered": reversed_sorted["fitted_count"],
                    "unsorted_fitted_reference": reference_unsorted["fitted_count"],
                    "unsorted_fitted_reordered": reversed_unsorted["fitted_count"],
                }

            for _ in range(12):
                shuffled_ids = list(probe_ids)
                rng.shuffle(shuffled_ids)
                shuffled_sorted = pack_truck(shuffled_ids, truck_type, sort_items=True)
                shuffled_unsorted = pack_truck(shuffled_ids, truck_type, sort_items=False)
                if pack_fingerprint(reference_unsorted) != pack_fingerprint(shuffled_unsorted):
                    return {
                        "verified": True,
                        "truck_type": truck_type,
                        "sample_size": sample_size,
                        "changed_without_sort": True,
                        "method": "shuffle",
                        "sorted_fitted_reference": reference_sorted["fitted_count"],
                        "sorted_fitted_reordered": shuffled_sorted["fitted_count"],
                        "unsorted_fitted_reference": reference_unsorted["fitted_count"],
                        "unsorted_fitted_reordered": shuffled_unsorted["fitted_count"],
                    }

    return {
        "verified": False,
        "truck_type": None,
        "sample_size": None,
        "changed_without_sort": False,
        "method": None,
        "sorted_fitted_reference": None,
        "sorted_fitted_reordered": None,
        "unsorted_fitted_reference": None,
        "unsorted_fitted_reordered": None,
    }


def print_overview():
    total_volume = sum(item_volume(item_id) for item_id in ALL_ITEM_IDS)
    total_weight = sum(item_weight(item_id) for item_id in ALL_ITEM_IDS)
    item_mix = Counter(item["name"] for item in ITEM_SPECS)
    print("货物总数:", len(ALL_ITEM_IDS))
    print("货物构成:", dict(item_mix))
    print("总体积:", total_volume)
    print("总重量:", total_weight)
    print()


def print_sort_switch_result(result):
    print("顺序开关验证:")
    print("关闭自动排序后能保留外部顺序影响:", result["verified"])
    print("sort_items=False 时结果随输入顺序变化:", result["changed_without_sort"])
    if result["truck_type"] is not None:
        print("验证车型:", TRUCK_TYPES[result["truck_type"]]["label"])
        print("验证样本规模:", result["sample_size"])
        print("验证方式:", result["method"])
        print("sort_items=True 装入件数对比:", result["sorted_fitted_reference"], result["sorted_fitted_reordered"])
        print("sort_items=False 装入件数对比:", result["unsorted_fitted_reference"], result["unsorted_fitted_reordered"])
    print()


def print_solution(label, evaluation):
    type1_count = evaluation["truck_type_counts"].get("type1", 0)
    type2_count = evaluation["truck_type_counts"].get("type2", 0)
    print(label)
    print("目标模式:", evaluation["objective_label"])
    print("目标值:", round(evaluation["objective"], 2))
    print("总成本:", evaluation["total_cost"])
    print("车型1车次:", type1_count)
    print("车型2车次:", type2_count)
    print("总车次:", evaluation["truck_count"])
    print("超出车数上限:", evaluation["truck_limit_exceeded"])
    print("未装货物数:", evaluation["total_unfitted_count"])
    print("是否可行:", evaluation["feasible"])
    print()


def print_truck_details(evaluation):
    for index, truck_result in enumerate(evaluation["truck_results"], start=1):
        print(f"第 {index} 辆车 - {truck_result['truck_label']}")
        print("  已分配件数:", truck_result["assigned_count"])
        print("  装入件数:", truck_result["fitted_count"])
        print("  未装件数:", truck_result["unfitted_count"])
        print("  体积利用率:", f"{truck_result['space_utilization']}%")
        print("  载重利用率:", f"{truck_result['load_utilization']}%")
        print("  货物构成:", dict(truck_result["item_mix"]))
        print()


def maybe_show_plots(evaluation):
    if not SHOW_PLOTS and not SAVE_PLOTS:
        return

    from py3dbp import Painter

    if SAVE_PLOTS:
        PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for truck_result in evaluation["truck_results"]:
        painter = Painter(truck_result["box"])
        figure = painter.plotBoxAndItems(
            title=truck_result["box"].partno,
            alpha=0.2,
            write_num=False,
            fontsize=6,
        )
        if SAVE_PLOTS:
            image_path = PLOT_OUTPUT_DIR / f"{truck_result['box'].partno}.png"
            figure.savefig(image_path, dpi=200, bbox_inches="tight")

        if SHOW_PLOTS:
            figure.show()
        else:
            figure.close('all')


def main():
    start_time = time.time()
    objective_mode = OBJECTIVE_MODE
    print("当前优化目标:", get_objective_label(objective_mode))
    print("车辆上限:", MAX_TRUCK_COUNT)
    print()
    print_overview()

    if RUN_SORT_SWITCH_VERIFICATION:
        sort_switch_result = verify_sort_switch()
    else:
        sort_switch_result = {
            "verified": False,
            "truck_type": None,
            "sample_size": None,
            "changed_without_sort": False,
            "method": None,
            "sorted_fitted_reference": None,
            "sorted_fitted_reordered": None,
            "unsorted_fitted_reference": None,
            "unsorted_fitted_reordered": None,
        }
    print_sort_switch_result(sort_switch_result)

    baselines = build_baselines(objective_mode=objective_mode)
    valid_baselines = []
    for baseline in baselines:
        print_solution(f"基线方案 - {baseline['name']}", baseline["evaluation"])
        if not baseline["evaluation"]["truck_limit_exceeded"]:
            valid_baselines.append(baseline)

    if not valid_baselines:
        raise ValueError(f"没有满足车数上限 {MAX_TRUCK_COUNT} 的基线方案")

    best_baseline = min(valid_baselines, key=lambda baseline: baseline["evaluation"]["objective"])
    best_baseline_fleet = best_baseline["fleet"]

    runs = []
    for run_index in range(SA_RUNS):
        seed = RANDOM_SEED + run_index
        run_result = simulated_annealing(best_baseline_fleet, seed, objective_mode=objective_mode)
        runs.append(run_result)
        print_solution(f"模拟退火第 {run_index + 1} 次运行 (seed={seed})", run_result["best_evaluation"])

    best_run = min(runs, key=lambda run: run["best_evaluation"]["objective"])
    average_cost = sum(run["best_evaluation"]["total_cost"] for run in runs) / len(runs)
    average_truck_count = sum(run["best_evaluation"]["truck_count"] for run in runs) / len(runs)
    final_evaluation = evaluate_fleet(
        best_run["best_fleet"],
        include_boxes=SHOW_PLOTS or SAVE_PLOTS,
        objective_mode=objective_mode,
    )

    print_solution("最终最优方案", final_evaluation)
    print("多次运行平均成本:", round(average_cost, 2))
    print("多次运行平均车次:", round(average_truck_count, 2))
    print()
    print_truck_details(final_evaluation)

    elapsed = time.time() - start_time
    print("整车缓存状态数:", len(EVALUATION_CACHE))
    print("单车缓存状态数:", len(PACK_TRUCK_CACHE))
    print("总运行时间:", round(elapsed, 2), "秒")

    maybe_show_plots(final_evaluation)


if __name__ == "__main__":
    main()
